# IMDb Review Summarization with Qwen3-0.6B

This small notebook loads the already-downloaded **Qwen3-0.6B base model**, selects a few IMDb reviews, and summarizes them.

In [12]:
# Install only if these packages are not already available.
%pip install -q transformers datasets accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
LOCAL_MODEL_DIR = Path("./base_model/Qwen3-0.6B")

# Prefer the explicit local folder used by the original notebook.
# Otherwise, read the model from the existing Hugging Face cache only.
MODEL_SOURCE = str(LOCAL_MODEL_DIR.resolve()) if LOCAL_MODEL_DIR.exists() else MODEL_NAME

print("Model source:", MODEL_SOURCE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_SOURCE,
    local_files_only=not LOCAL_MODEL_DIR.exists(),
)

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_SOURCE,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
    local_files_only=not LOCAL_MODEL_DIR.exists(),
)

model.eval()
print("Model loaded successfully.")

Model source: E:\StudyFile\2026-T2\COMP9444-Neural Networks, Deep Learning\Group Project\Qwen3_0.6B_IMDB\base_model\Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model loaded successfully.


In [14]:
# Load and select a few IMDb reviews.
dataset = load_dataset("stanfordnlp/imdb", split="test")
reviews = dataset.shuffle(seed=42).select(range(10))

for i, item in enumerate(reviews, start=1):
    print(f"Review {i}:")
    print(item["text"][:800], "\n")

Review 1:
<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the characters with such depth and gentleness!<br /><br />I recognized the Angry sister; the Runaway sister and the sister in Denial. I recognized the Abusive Husband and why he was there and then the Father, oh oh the Father... all superbly played. I also recognized myself and this  

Review 2:
This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in the early 1950's, and spawne

In [15]:
def summarize_review(review: str) -> str:
    messages = [
        {
            "role": "user",
            "content": (
                "Summarize the following movie review in one or two clear sentences. "
                "Focus on the reviewer's main opinion and key reasons.\n\n"
                f"Review:\n{review}"
            ),
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()



In [16]:
import pandas as pd

results = []

for i, item in enumerate(reviews, start=1):
    original_review = item["text"]
    summary = summarize_review(original_review)

    results.append({
        "review_id": i,
        "original_review": original_review,
        "summary": summary,
    })

df = pd.DataFrame(results)
df.to_csv("final_result/imdb_review_summaries.csv", index=False, encoding="utf-8-sig")

print("Saved to imdb_review_summaries.csv")
df

Saved to imdb_review_summaries.csv


,review_id,original_review,summary
0,1,<br /><br />When I unsuspectedly rented A Thou...,"The movie ""A Thousand Acres"" was a deeply movi..."
1,2,This is the latest entry in the long series of...,"The reviewer recommends ""O.S.S.117:Cairo,Nest ..."
2,3,This movie was so frustrating. Everything seem...,The reviewer expresses frustration with the mo...
3,4,"I was truly and wonderfully surprised at ""O' B...","The reviewer strongly recommends ""O' Brother, ..."
4,5,This movie spends most of its time preaching t...,The reviewer criticizes the movie for not havi...
5,6,After a very long time Marathi cinema has come...,The reviewer recommends this Marathi movie as ...
6,7,"This is a really sad, and touching movie! It d...",The reviewer expresses a positive opinion abou...
7,8,Don't pay any attention to the rave reviews of...,"The reviewer strongly criticizes the film, cal..."
8,9,Porn legend Gregory Dark directs this cheesy h...,"Gregory Dark's cheesy horror flick, *Glen Jaco..."
9,10,This was a great movie. Something not only for...,"The movie was a great experience, highlighting..."
